# 03 — Análisis de features

El objetivo es evaluar conscientemente las variables originales antes de crear nuevas. El análisis parte de las decisiones consolidadas en el EDA y mantiene separadas la descripción, la selección y las transformaciones posteriores.

## Punto de partida

El modelo principal estimará el riesgo inmediatamente después de que una reserva sea confirmada y registrada. Se excluyen las variables que revelan el resultado y aquellas cuyo valor final puede generarse después de ese momento. Los nulos y casos atípicos se validan, pero todavía no se imputan ni transforman.

## Criterio experimental

Toda decisión basada en rendimiento debe evaluarse dentro de la estrategia de validación definida y sin utilizar el conjunto de evaluación final.

## Configuración y carga reproducible

Se reutilizan las funciones de `src.data` para cargar, integrar y normalizar los archivos. La carga prioriza los CSV locales y utiliza las URL Raw sólo cuando no están disponibles.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)

In [2]:
def find_project_root(start_path=None):
    """Busca la raíz del repositorio a partir del directorio actual."""
    start = Path(start_path or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir() and (candidate / "data" / "raw").is_dir():
            return candidate

    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz local del proyecto. "
        "Para ejecutar este notebook de forma remota debe estar disponible la carpeta src."
    )

project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
print(f"Raíz del proyecto: {PROJECT_ROOT}")

Raíz del proyecto: C:\Code\Vialesoft\Vialesoft_Devlab\Mini_Proyectos\Machine_Learning\Obligatorio2026


In [3]:
from src.data import (
    combine_hotel_datasets,
    load_csv_with_fallback,
    normalize_text_values,
    split_features_target,
)

H1_URL = "https://raw.githubusercontent.com/Lithium582/Obligatorio2026/refs/heads/main/data/raw/H1.csv"
H2_URL = "https://raw.githubusercontent.com/Lithium582/Obligatorio2026/refs/heads/main/data/raw/H2.csv"

h1_df = load_csv_with_fallback(RAW_DATA_DIR / "H1.csv", H1_URL)
h2_df = load_csv_with_fallback(RAW_DATA_DIR / "H2.csv", H2_URL)
bookings_df = combine_hotel_datasets(h1_df, h2_df)
bookings_df = normalize_text_values(bookings_df, null_labels=("NULL",))

print(f"Dataset integrado: {bookings_df.shape[0]:,} filas y {bookings_df.shape[1]} columnas.")

Dataset integrado: 119,390 filas y 32 columnas.


## Validación de la base analítica

Las comprobaciones siguientes protegen los supuestos establecidos durante el EDA. Si cambia el esquema, aparece un target inválido o surge un nulo en una columna inesperada, el notebook debe informarlo antes de continuar.

In [4]:
TARGET_COLUMN = "IsCanceled"
DIRECT_LEAKAGE_COLUMNS = ["ReservationStatus", "ReservationStatusDate"]
POST_CONFIRMATION_COLUMNS = ["AssignedRoomType", "BookingChanges"]
COLUMNS_TO_EXCLUDE = [*DIRECT_LEAKAGE_COLUMNS, *POST_CONFIRMATION_COLUMNS]

required_columns = {TARGET_COLUMN, *COLUMNS_TO_EXCLUDE}
missing_required_columns = sorted(required_columns.difference(bookings_df.columns))
if missing_required_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_required_columns}")

if bookings_df[TARGET_COLUMN].isna().any():
    raise ValueError("La variable objetivo contiene valores nulos.")

observed_target_values = set(bookings_df[TARGET_COLUMN].unique())
if observed_target_values != {0, 1}:
    raise ValueError(
        f"Se esperaban los valores de target {{0, 1}} y se encontraron {observed_target_values}."
    )

expected_rows = len(h1_df) + len(h2_df)
if len(bookings_df) != expected_rows:
    raise ValueError(
        f"La integración produjo {len(bookings_df):,} filas; se esperaban {expected_rows:,}."
    )

print("Esquema, cantidad de filas y variable objetivo validados.")

Esquema, cantidad de filas y variable objetivo validados.


### Validación de nulos conocidos

El EDA encontró nulos únicamente en `Company`, `Agent`, `Country` y `Children`. Se permite que esas columnas conserven nulos para que su tratamiento se implemente más adelante dentro del pipeline. Cualquier columna nula adicional se considera un cambio inesperado.

In [5]:
EXPECTED_NULL_COLUMNS = {"Company", "Agent", "Country", "Children"}
null_counts = bookings_df.isna().sum()
observed_null_columns = set(null_counts[null_counts.gt(0)].index)
unexpected_null_columns = sorted(observed_null_columns - EXPECTED_NULL_COLUMNS)

if unexpected_null_columns:
    raise ValueError(
        f"Aparecieron nulos en columnas no previstas: {unexpected_null_columns}"
    )

null_validation_summary = pd.DataFrame(
    {
        "null_rows": null_counts.loc[sorted(EXPECTED_NULL_COLUMNS)],
    }
)
null_validation_summary["null_percentage"] = (
    null_validation_summary["null_rows"] / len(bookings_df) * 100
)
display(null_validation_summary.sort_values("null_rows", ascending=False))
print("No se encontraron nulos en columnas inesperadas.")

,null_rows,null_percentage
Company,112593,94.306893
Agent,16340,13.686238
Country,488,0.408744
Children,4,0.003350


No se encontraron nulos en columnas inesperadas.


## Exclusiones definidas antes del análisis

Estas columnas no se descartan por una asociación estadística débil, sino porque no son admisibles para el momento de predicción. Se eliminan sólo de `X`; el dataset integrado se conserva intacto para auditoría.

In [6]:
exclusion_summary = pd.DataFrame(
    [
        {
            "column": "ReservationStatus",
            "reason": "Describe el desenlace y determina directamente IsCanceled.",
        },
        {
            "column": "ReservationStatusDate",
            "reason": "Registra la fecha en la que se produjo el desenlace.",
        },
        {
            "column": "AssignedRoomType",
            "reason": "La asignación final puede ocurrir después de confirmar la reserva.",
        },
        {
            "column": "BookingChanges",
            "reason": "Acumula modificaciones posteriores a la confirmación.",
        },
    ]
)
display(exclusion_summary)

,column,reason
0,ReservationStatus,Describe el desenlace y determina directamente...
1,ReservationStatusDate,Registra la fecha en la que se produjo el dese...
2,AssignedRoomType,La asignación final puede ocurrir después de c...
3,BookingChanges,Acumula modificaciones posteriores a la confir...


In [7]:
X_candidates, y = split_features_target(
    bookings_df,
    target_col=TARGET_COLUMN,
    columns_to_drop=COLUMNS_TO_EXCLUDE,
)

remaining_excluded_columns = sorted(
    set(COLUMNS_TO_EXCLUDE).intersection(X_candidates.columns)
)
if remaining_excluded_columns:
    raise ValueError(
        f"Las columnas excluidas continúan en X: {remaining_excluded_columns}"
    )

if TARGET_COLUMN in X_candidates.columns:
    raise ValueError("La variable objetivo continúa presente en X.")

print(f"Features candidatas: {X_candidates.shape[1]}")
display(pd.DataFrame({"feature": X_candidates.columns}))

Features candidatas: 27


,feature
0,HotelType
1,LeadTime
2,ArrivalDateYear
3,ArrivalDateMonth
4,ArrivalDateWeekNumber
5,ArrivalDateDayOfMonth
6,StaysInWeekendNights
7,StaysInWeekNights
8,Adults
9,Children


La base de Feature Analysis conserva una fila por reserva y todas las variables admisibles después de la confirmación. Los nulos conocidos permanecen sin imputar y los casos atípicos no se alteran. El próximo paso será clasificar las features por significado —binarias, categóricas, identificadores, calendario, conteos e importes— antes de medir su relación con el target.